## Milestone 3
### Jennifer Tsang, Nicole Link

In [ ]:
import os
from pathlib import Path
import pandas as pd
import duckdb
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.retrievers import EnsembleRetriever
import pickle
import transformers
from transformers import pipeline
import torch
from langchain_groq import ChatGroq

In [3]:
if Path.cwd().name != "DSCI_575_project_jt8919_nicolelink33":
    project_root = Path.cwd().parent 
    os.chdir(project_root)
    
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/Nicole/MDS/575/DSCI_575_project_jt8919_nicolelink33


In [50]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from src.utils import simple_tokenize, display_results
from src.bm25 import search_bm25_with_scores
from src.semantic import search_semantic_with_scores
from src.hybrid import get_hybrid_retriever
from src.rag_pipeline import build_context, get_prompt_template, get_rag_chain

In [ ]:
DATA_DIR = Path("data")
CATEGORY = "Arts_Crafts_and_Sewing"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"
bm25_path = "models/bm25_metadata_index_big.pkl"
semantic_path = "models/faiss_index_big"
OUTPUT = 'data/processed/stratified_sample_big.parquet'

In [6]:
print(REVIEWS_URL)

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Arts_Crafts_and_Sewing.jsonl.gz


## 0. Loading and Merging the Datasets

### Data Loading

In [7]:
c2 = duckdb.connect()

In [8]:
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0,Received damaged. Has hole in mold!,[[VIDEOID:2a7ad2a91afb1e17ff4a1c143f7e10a2]] R...,[{'small_image_url': 'https://m.media-amazon.c...,B095RKB9N3,B095RXT585,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1661111719157,0,True
1,3.0,3rd one arrived scratched/dented.,3rd one that arrived damaged! I give up!,[{'small_image_url': 'https://m.media-amazon.c...,B08PNNKNSQ,B07QXN7TMP,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1655614875170,0,True
2,1.0,Abominable. NOT COLOR SHIFT! False Advertising!,These are regular mica powders! One appears t...,[{'small_image_url': 'https://m.media-amazon.c...,B094DH11SH,B094DH11SH,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1654510930828,0,True
3,2.0,Used 2x in 6 weeks dry as a bone.,[[VIDEOID:8644a01103c90ab8f83af816b83881be]] L...,[{'small_image_url': 'https://m.media-amazon.c...,B089YSSFD6,B087D548ZY,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1648170476726,0,True
4,1.0,Replaced & still Defective stamps!,Lowered from a 3 star review to a 1 star bc th...,[{'small_image_url': 'https://m.media-amazon.c...,B08Z47RG56,B08Z47RG56,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1644619507972,1,True


In [9]:
# executes right over the internet -- i just want five rows to preview so doesn't take long
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Amazon Home,Trapp Home Fragrance Wax Melts - No. 13 Bob's ...,4.4,108,[Includes (2) No. 13 Bob's Flower Shoppe Trapp...,[The No. 13 Bob's Flower Shoppe Home Fragrance...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Trapp,"[Arts, Crafts & Sewing, Crafting, Candle Makin...","{'Product Dimensions': '""6.75 x 3.4 x 1 inches...",B013W3MPCW,None,None,<NA>
1,"Arts, Crafts & Sewing",JESEP YONG 4packs Plastic Organizer Box 24 Gri...,4.3,103,[MATERIAL & COLOR: These storage boxes are mad...,[SIZE & PACKING: the outer box size is 7.5'' L...,12.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'craft organizers and storage', 'ur...",JESEP YONG,"[Arts, Crafts & Sewing, Organization, Storage ...","{'Brand': '""JESEP YONG""', 'Color': '""Clear""', ...",B09B59XWTG,None,None,<NA>
2,Pet Supplies,"JIESHU 2-Pack Foam Holders for centerpieces, B...",3.6,15,[The 6 suction cups at the bottom of the foam ...,[Size: 9.5 * 4.5 * 2.7 inches Ideal for manual...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],JIESHU,"[Arts, Crafts & Sewing, Crafting, Floral Arran...","{'Product Dimensions': '""9.5 x 4.5 x 2.7 inche...",B0915N2NX1,None,None,<NA>
3,"Arts, Crafts & Sewing",Metal Swivel Lobster Claw Clasp Split Ring Key...,3.2,35,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],NaN,"[Arts, Crafts & Sewing, Beading & Jewelry Maki...","{'Product Dimensions': '""4.1 x 3.7 x 0.6 inche...",B008SNYJSU,None,None,<NA>
4,"Arts, Crafts & Sewing",Sizzix Birdcage Favor Thinlits Gift Box by Dav...,4.6,464,[Thinlits dies allow you to cut intricate desi...,[Thinlits create dazzling detailed shapes for ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sizzix,"[Arts, Crafts & Sewing, Scrapbooking & Stampin...","{'Product Dimensions': '""5.38 x 4.38 x 0.13 in...",B07Z9FH9PR,None,None,<NA>


#### Download and convert to parquet

In [10]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{REVIEWS_URL}')  LIMIT 20000)
      TO 'data/raw/reviews_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

In [11]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 20000)
      TO 'data/raw/meta_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

#### Merge the two files

In [13]:
c2.execute("""
    COPY (
        SELECT r.rating, r.title, r.text, r.parent_asin, r.verified_purchase,
           r.helpful_vote,
           m.title AS product_title, m.price,
           m.average_rating, m.main_category, m.store
        FROM read_parquet('data/raw/reviews_raw.parquet') r
        LEFT JOIN read_parquet('data/raw/meta_raw.parquet') m USING (parent_asin)
    )
    TO 'data/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [14]:
c2.execute(f"SELECT * FROM read_parquet('data/merged.parquet')").df()


,rating,title,text,parent_asin,verified_purchase,helpful_vote,product_title,price,average_rating,main_category,store
0,5.0,"Beautiful, heavily textured vinyl",I really like this assortment of metallic viny...,B0BW3X9XT4,False,0,Holographic Vinyl Black Rainbow Permanent Viny...,13.90,4.6,Amazon Home,GIRAFVINYL
1,2.0,Too small and seem to be better for rings,I've had a hard time using these for bracelets...,B0058EDF8C,True,0,"Eurotool Pliers, Silver",21.99,4.1,"Arts, Crafts & Sewing",EURO TOOL
2,5.0,Awesome product.,I absolutely loved being able to buy feathers ...,B07NP9VFZ3,True,1,wanjin Duck Goose Feathers Trim Fringe Craft F...,14.99,4.6,"Arts, Crafts & Sewing",Wanjin
3,5.0,Very true to color in picture. Fast shipping. ...,Beautiful beads very true to picture. Wish I h...,B00J9F6DEO,True,5,BEADNOVA 8mm White Clear Crystal Quartz Gemsto...,11.39,4.7,"Arts, Crafts & Sewing",BEADNOVA
4,5.0,Five Stars,I love this small size sketch pad. Perfect si...,B088VVG9HN,True,0,"Strathmore 300 Series Sketch Paper Pad, Glue B...",24.61,4.7,"Arts, Crafts & Sewing",Strathmore
...,...,...,...,...,...,...,...,...,...,...,...
19995,5.0,Five Stars,Great pen set!! I would buy this over and over.,B01GLS0C2K,True,1,NaN,NaN,NaN,NaN,NaN
19996,5.0,Wow,Beautiful product. Very secure. Gave a few awa...,B017K4AFF8,True,0,NaN,NaN,NaN,NaN,NaN
19997,5.0,Perfect for my needs.,Melts very easily. Great value at this size,B001QMY0RU,True,0,NaN,NaN,NaN,NaN,NaN
19998,5.0,Great selection of zippers for the price!,so far I’m very well pleased with the value I ...,B0756TPB4S,True,2,NaN,NaN,NaN,NaN,NaN


### Create Stratified Sample

In [41]:
SAMPLE_PER_STRATUM = 10000    # reviews to keep per stratum cell (floor guarantee)
MIN_TEXT_LEN       = 0    # drop near-empty reviews (chars)
SHORT_MAX          = 100   # short: text < SHORT_MAX chars
MEDIUM_MAX         = 500   # medium: SHORT_MAX ≤ text < MEDIUM_MAX chars
                        # long: text ≥ MEDIUM_MAX chars

OUTPUT = 'data/processed/stratified_sample_big.parquet'

c2.execute(f"""
COPY (
WITH labelled AS (
    SELECT
        text, rating, verified_purchase, helpful_vote,
        parent_asin,
        product_title, price,
        average_rating, main_category,

        CASE
            WHEN average_rating >= 4.6 THEN '4.6-5.0'
            WHEN average_rating >= 4.4 THEN '4.4-4.5'
            WHEN average_rating >= 4.1 THEN '4.1-4.3'
            WHEN average_rating >= 3.7 THEN '3.7-4.0'
            WHEN average_rating >= 3.1 THEN '3.1-3.6'
            ELSE                              '<=3.0'
        END AS rating_bucket,

        CASE
            WHEN LENGTH(text) < {SHORT_MAX}  THEN 'short'
            WHEN LENGTH(text) < {MEDIUM_MAX} THEN 'medium'
            ELSE                                    'long'
        END AS len_tier

    FROM read_parquet('data/merged.parquet')
),

-- Step 1: one review per product per cell: prevents a single popular product flooding a stratum, you can redefine
one_per_product AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY rating_bucket, len_tier, parent_asin
            ORDER BY helpful_vote DESC, random()
        ) AS product_rank
    FROM labelled
),

-- Step 2: rank within each stratum cell, helpful reviews first
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY rating_bucket, len_tier
            ORDER BY helpful_vote DESC, random()
        ) AS stratum_rank
    FROM one_per_product
    WHERE product_rank = 1
)

SELECT * EXCLUDE (product_rank, stratum_rank)
FROM ranked
WHERE stratum_rank <= {SAMPLE_PER_STRATUM}
)
TO '{OUTPUT}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

### Convert Parquet to LangChain Documents

In [42]:
# 1. Load your newly minted stratified sample
df_sample = pd.read_parquet(OUTPUT)

# 2. Fill any random missing text values to prevent concatenation crashes
text_columns = ['product_title', 'main_category', 'text']
for col in text_columns:
    if col in df_sample.columns:
        df_sample[col] = df_sample[col].fillna("")

# 3. Engineer the hybrid page_content string
df_sample['page_content'] = (
    "Product Title: " + df_sample['product_title'] + "\n" +
    "Category: " + df_sample['main_category'] + "\n" +
    "Review Text: " + df_sample['text']
)

# 4. Drop the redundant text columns so they don't clutter the metadata
df_clean = df_sample.drop(columns=['product_title', 'main_category', 'text'])

# 5. Convert the DataFrame into LangChain Documents
documents = []
for _, row in df_clean.iterrows():
    # The engineered string becomes the searchable content
    content = row['page_content']
    
    # Every other column (rating, price, helpful_vote, rating_bucket, len_tier) 
    # gets scooped up into the metadata dictionary!
    metadata = row.drop('page_content').to_dict()
    
    # Create and append the Document
    doc = Document(page_content=content, metadata=metadata)
    documents.append(doc)

print(f"Successfully created {len(documents)} LangChain Documents!")
print("\n--- Let's peek at the first one ---")
print(f"CONTENT:\n{documents[0].page_content}")
print(f"METADATA:\n{documents[0].metadata}")

Successfully created 16809 LangChain Documents!

--- Let's peek at the first one ---
CONTENT:
Product Title: Plaid Patricia Nimocks Clear Acrylic Sealer (12-Ounce), CS200305 Gloss
Category: Arts, Crafts & Sewing
Review Text: Using this for the Kindness Rocks Project here. Good sealer as long as you don't spray too thickly.
METADATA:
{'rating': 4.0, 'verified_purchase': True, 'helpful_vote': 5, 'parent_asin': 'B07NLL1N36', 'price': 13.99, 'average_rating': 4.6, 'rating_bucket': '4.6-5.0', 'len_tier': 'short'}


## 1. Data Overview:

In [35]:
# The number of products
c2.execute("""
    SELECT COUNT(*) 
    FROM read_parquet('data/merged.parquet') 
""").df()

,count_star()
0,20000


In [37]:
# The number of products with reviews and ratings
c2.execute("""
    SELECT COUNT(*) 
    FROM read_parquet('data/merged.parquet') 
    WHERE text IS NOT NULL 
      AND average_rating IS NOT NULL
""").df()

,count_star()
0,1339


In [47]:
print(df_sample.info())
print(df_sample.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 16809 entries, 0 to 16808
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   text               16809 non-null  str    
 1   rating             16809 non-null  float64
 2   verified_purchase  16809 non-null  bool   
 3   helpful_vote       16809 non-null  int64  
 4   parent_asin        16809 non-null  str    
 5   product_title      16809 non-null  str    
 6   price              875 non-null    float64
 7   average_rating     1045 non-null   float64
 8   main_category      16809 non-null  str    
 9   rating_bucket      16809 non-null  str    
 10  len_tier           16809 non-null  str    
 11  page_content       16809 non-null  str    
dtypes: bool(1), float64(3), int64(1), str(7)
memory usage: 13.8 MB
None
['text', 'rating', 'verified_purchase', 'helpful_vote', 'parent_asin', 'product_title', 'price', 'average_rating', 'main_category', 'rating_bucket', 'len_tie

#### Rating Distribution

In [48]:
# EDA example duckdb Rating distribution
c2.execute("""
    SELECT
        rating,
        COUNT(*) AS count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percent
    FROM read_parquet('data/merged.parquet')
    GROUP BY 1
    ORDER BY 1
""").df()

,rating,count,percent
0,1.0,895,4.47
1,2.0,676,3.38
2,3.0,1259,6.30
3,4.0,2503,12.52
4,5.0,14667,73.33


## 2. Model Building
- Builds **BM25 Search** model and saves it as a pickle file locally into the model's directory.
- Builds **Semantic Search** model and saves the index locally into the model's directory. 

### BM25 Search

In [49]:
# Pull documents one by one, tokenize them, and build the math model
retriever = BM25Retriever.from_documents(
    documents,
    preprocess_func=simple_tokenize
)

retriever.k = 3

# Save the BM25 index locally 
with open(bm25_path, 'wb') as f:
    pickle.dump(retriever, f)
    
print(f"Index built and safely saved to {bm25_path}!")

Index built and safely saved to models/bm25_metadata_index_big.pkl!


#### BM25 Retrieval Score
- Testing the saved BM25 model to make sure it works!

In [50]:
query = "green yarn"
print(f"\nTesting Loaded BM25 Search with query: '{query}'")

# Open the pickle file in 
with open(bm25_path, 'rb') as f:
    loaded_bm25_retriever = pickle.load(f)

# Run search using loaded model
bm25_scored_results = search_bm25_with_scores(loaded_bm25_retriever, query, k=3)
display_results(bm25_scored_results, title="Loaded BM25 Results", score_label="BM25 Score")


Testing Loaded BM25 Search with query: 'green yarn'
Loaded BM25 Results
#1 | BM25 Score: 13.361
Product: Lion Brand Yarn 595-205 Color Waves Yarn, Green Apple
Rating:  ⭐⭐⭐⭐⭐ (5/5)
Review:  wife loved it.
--------------------------------------------------
#2 | BM25 Score: 11.877
Product: T-Shirt Yarn Recycled 130 Yards 1.5 lb Bulky Yarn│Jersey Yarn│Fabric Yarn │T Shirt Yarn for Crochet │ Knitting Tshirt Yarn │ Home Decor DYI Supply │ Recycled Yarn │Trapillo (Apple Green)
Rating:  ⭐⭐⭐⭐ (4/5)
Review:  Using for mask straps. Working great. I just wish that it were a little more soft and that it came in black as well as white!
--------------------------------------------------
#3 | BM25 Score: 10.000
Product: 100% Cotton Loom Warp Thread (Green), 8/4 Warp Yarn (800 Yards), Perfect for Weaving: Carpet, Tapestry, Rug, Blanket or Pattern - Warping Thread for Any Loom
Rating:  ⭐⭐⭐⭐⭐ (5/5)
Review:  Very nice but wrong for my project.
--------------------------------------------------


### Semantic Search

In [51]:
model = SentenceTransformer("all-MiniLM-L6-v2")
content_list = [doc.page_content for doc in documents]

document_embeddings = model.encode(content_list, show_progress_bar=True)
print(f"Successfully created {len(document_embeddings)} embeddings!")

Batches: 100%|██████████| 526/526 [01:14<00:00,  7.08it/s]

Successfully created 16809 embeddings!


In [52]:
# Format the data for LangChain
text_embedding_pairs = list(zip(content_list, document_embeddings.tolist()))
metadata_list = [doc.metadata for doc in documents]

# Query embedder
hf_query_embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Build the Vector Store 
vector_store = FAISS.from_embeddings(
    text_embeddings=text_embedding_pairs,
    embedding=hf_query_embedder,
    metadatas=metadata_list
)

# Save it to disk locally
vector_store.save_local(semantic_path)
print(f"FAISS Index successfully saved to {semantic_path}!")

FAISS Index successfully saved to models/faiss_index_big!


#### Semantic Retrieval Score
- Testing the saved Semantic Search model to make sure it works!

In [53]:
print(f"\nTesting Loaded Semantic Search with query: '{query}'")

loaded_vector_store = FAISS.load_local(
    folder_path=semantic_path,
    embeddings=hf_query_embedder,
    allow_dangerous_deserialization=True
)

semantic_scored_results = search_semantic_with_scores(loaded_vector_store, query, k=3)
display_results(semantic_scored_results, title="FAISS Results", score_label="L2 Distance")


Testing Loaded Semantic Search with query: 'green yarn'
FAISS Results
#1 | L2 Distance: 0.681
Product: 
Rating:  ⭐⭐⭐⭐ (4/5)
Review:  OK, I have to start by admitting that I love to work with shaded yarn.  (not so many yarn ends to tuck in.....)  The colors in this yarn would make a lovely baby blanket or sweater.  I am actually...
--------------------------------------------------
#2 | L2 Distance: 0.711
Product: 
Rating:  ⭐⭐⭐⭐⭐ (5/5)
Review:  I was attracted to this yarn by the soft, shaded colors.  It makes a very attractive garment.  It really gives the finished product a lovely, special look.  I used it to make a shawl, and I am very...
--------------------------------------------------
#3 | L2 Distance: 0.727
Product: 
Rating:  ⭐⭐⭐⭐⭐ (5/5)
Review:  Love this! Luxurious merino; SOFT,  plush, knits up beautifully with nice stitch definition. Very few flying fibers, and on US 15 birch needles I’ve had no splitting issues. The consistency is...
---------------------------------------

## Comparison with New LLM Model

In [54]:
query1 = "sewing machine"
query2 = "acrylic paint"
query3 = "Singer 5532"
query4 = "Animal stickers"
query5 = "Purple gel pen"
query6 = "cute embroidery patterns"
query7 = "yarn made of natural fibers"
query8 = "best paint for wood"
query9 = "what's the best type of yarn to make clothes with"
query10 = "what is a good type of yarn for a beginner"

Originally chose `HuggingFace model`: `Qwen/Qwen2.5-0.5B`

In [39]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B",
    torch_dtype = torch.float16,
    device="mps",
    trust_remote_code=True,
    max_new_tokens=128,
    do_sample=True,
)

llm = HuggingFacePipeline(pipeline=generator)

Device set to use mps


In [43]:
old_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, llm)

old_ensemble_answer = old_rag_chain.invoke(query1)

/Users/Nicole/miniforge3/envs/575-app/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [44]:
print(old_ensemble_answer)

Human: 
    You are a helpful Amazon shopping assistant.
        Answer the question using ONLY the following context (real product reviews + metadata).
        Always cite the product ASIN when possible.

    Context:
    Product ASIN: B07YZV2485
Title: Unknown Product
Rating: 5.0/5
Review: I can’t speak for how good this would be in a machine, but it works very well for hand sewing. A nice selection of good quality colors.

Product ASIN: B087PBSF55
Title: Unknown Product
Rating: 5.0/5
Review: Great for the beginner or the person just taking up sewing again.  This small sewing machine has many of the features of a full-size machine.  It offers a variety of stitches, sews backward or forward, operates with bobbins, and uses either a power adapter or batteries.  It runs smoothly delivering a smooth, regular stitch.  It has a foot pedal for those who are used to having one.  Really pleased wit this energetic sewing machine!

Product ASIN: B097FPLKP2
Title: SINGER | 4423 Heavy Duty Sewing

### New Model:
For our new model, we chose to use the GroqChat model "llama-3.1-8b-instant". 

In [53]:
new_llm = ChatGroq(model="llama-3.1-8b-instant")

In [54]:
new_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, new_llm)

new_ensemble_answer = new_rag_chain.invoke(query1)
print(new_ensemble_answer)

It seems like you're looking for information about sewing machines. 

Based on the given reviews, here are some sewing machine recommendations:

For beginners or those taking up sewing again, consider the **B087PBSF55** sewing machine. It's a small machine with many features of a full-size machine, including various stitches and a foot pedal.

For hand sewing, you can use **B07YZV2485**. It's a good selection of colors and seems to work well for hand sewing.

If you're looking for a travel sewing machine, consider **B08LYPVBJ8** - a travel sewing machine bag with a cover that's perfect for storing and transporting your sewing machine.

However, if you're looking to purchase a sewing machine, be cautious of the **B097FPLKP2** SINGER heavy-duty sewing machine. While it's received some positive reviews, some customers have reported receiving a used machine despite it being packaged as new.

It's also worth noting that the **B08Y5NW6XN** Magicfly Mini Sewing Machine has received excellent 

#### Query 3

In [55]:
old_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, llm)

old_ensemble_answer = old_rag_chain.invoke(query3)
print(old_ensemble_answer)

/Users/Nicole/miniforge3/envs/575-app/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


Human: 
    You are a helpful Amazon shopping assistant.
        Answer the question using ONLY the following context (real product reviews + metadata).
        Always cite the product ASIN when possible.

    Context:
    Product ASIN: B00WI1L58A
Title: SINGER 57261 Vintage Sewing Baskets, Large, Pink/Black
Rating: 5.0/5
Review: My teenager "adopted" my sewing basket. So when i started looking for a new one, i saw the name Singer, and i knew it would be quality. I was right! Bonus! It even came with items as a starter kit! Lots of room for small and large items, and the fabric is adorable. Ill have this a long time, or at least till my 7 year old turns 16 and "adopts" this one, lol.

Product ASIN: B0C2PRCYYV
Title: Unknown Product
Rating: 5.0/5
Review: This fits my Singer sewing machine perfectly. I like the little pockets on the front. I love it.

Product ASIN: B001V8JOFW
Title: Unknown Product
Rating: 5.0/5
Review: I have an early 1970's Singer 20U-13 machine.  You can no longer fin

In [56]:
new_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, new_llm)

new_ensemble_answer = new_rag_chain.invoke(query3)
print(new_ensemble_answer)

The Singer 5532 Heavy Duty Extra-High Sewing Speed Portable Sewing Machine with Metal Frame and Stainless Steel Bedplate has a high-speed sewing capability of 1,100 stitches per minute.


#### Query 5

In [57]:
old_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, llm)

old_ensemble_answer = old_rag_chain.invoke(query5)
print(old_ensemble_answer)

/Users/Nicole/miniforge3/envs/575-app/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


Human: 
    You are a helpful Amazon shopping assistant.
        Answer the question using ONLY the following context (real product reviews + metadata).
        Always cite the product ASIN when possible.

    Context:
    Product ASIN: B0B4G5JVLW
Title: Unknown Product
Rating: 5.0/5
Review: I have reviewed and purchased many different kinds of gel and glitter pens. By far these are the best in recent memory. Smooth writing and good colors. Much like a ballpoint pen. Very nice!

Product ASIN: B09N3RY7X6
Title: Unknown Product
Rating: 5.0/5
Review: Hotep family. What I like about these gel pens is they have gel glitter within. In my mixed media art sometimes, I like to start with a line or two that's very accurate and then more Pollack art. I have carpal tunnel. so, I can't hold a pen too long, but I liked the ease I puts down a nice line or BLOB, to add to my artistic soul song. I recommend my friend; these are a nice set of 18 colors to blend. Thank you for looking at my review. I hop

In [58]:
new_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, new_llm)

new_ensemble_answer = new_rag_chain.invoke(query5)
print(new_ensemble_answer)

Based on the provided reviews, I couldn't find any explicit mention of "purple gel pen". However, some reviews mention gel pens with specific colors.

In the review for Product ASIN: B08PP4C9DT, the reviewer mentions that the gel pens have "very vivid" colors. Unfortunately, the specific color is not mentioned.

In the review for Product ASIN: B08ZXS1DN2, the reviewer mentions that the gel pens have "bold" colors, but again, the specific color is not mentioned.

In the review for Product ASIN: B09D2SFW5Y, the product is a "White Gel Pen Set" and does not seem to match the color "purple".

If you are looking for a purple gel pen, I would recommend searching Amazon for "purple gel pen" to find relevant results.


### Query 7

In [59]:
old_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, llm)

old_ensemble_answer = old_rag_chain.invoke(query7)
print(old_ensemble_answer)

Human: 
    You are a helpful Amazon shopping assistant.
        Answer the question using ONLY the following context (real product reviews + metadata).
        Always cite the product ASIN when possible.

    Context:
    Product ASIN: B07X5MCHWJ
Title: Unknown Product
Rating: 5.0/5
Review: good basic yarn. it has different textured fibers mixed in so its not as smooth as some other yarns. I like the natural neutral color.

Product ASIN: B09ZQKYGMV
Title: Unknown Product
Rating: 4.0/5
Review: This yarn is very soft and of very good quality. It has been easy to work with. Pricey, but worth it.

Product ASIN: B09L5MVFP2
Title: Unknown Product
Rating: 5.0/5
Review: This is lovely yarn, with a pretty &#34;natural look&#34; and a good feel which it owes to the wool blend  It gives you the warmth and feel of wool with much of the convenience of synthetic yarns.  I am using it for a scarf.  It works well and would be a good choice for many projects, just follow the easy care directions and y

In [60]:
new_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, new_llm)

new_ensemble_answer = new_rag_chain.invoke(query7)
print(new_ensemble_answer)

Based on the provided reviews, the following products are made of natural fibers:

1. Product ASIN: B09L5MVFP2 - This yarn is a wool blend, which means it contains natural fibers from wool.
2. Product ASIN: B0BWSF1HPY - Although the review mentions it feels like what you'd use to make sweaters, it doesn't explicitly state the type of fibers used. However, the product title mentions "Cotton" which is a natural fiber.
3. Product ASIN: B07XM148NR - The review mentions that the yarn generates heat when exposed to light, which is due to the reaction of long and short fibers to light. This implies that the yarn contains natural fibers that react to light.

Additionally, the reviews for Product ASIN: B0B3RK8Q1B mention "angora yarns" which are typically made from natural fibers, specifically angora fibers.


#### Query 9

In [61]:
old_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, llm)

old_ensemble_answer = old_rag_chain.invoke(query9)
print(old_ensemble_answer)

Human: 
    You are a helpful Amazon shopping assistant.
        Answer the question using ONLY the following context (real product reviews + metadata).
        Always cite the product ASIN when possible.

    Context:
    Product ASIN: B0BXS2C7FT
Title: Unknown Product
Rating: 4.0/5
Review: This is a very nice cotton yarn.  It's weight is more like a mercerized cotton though it is not mercerized.  So it will not crochet up in the same way as sugar 'n cream.  Crocheting single crochet, same hook (H) I had about 10 stitches with this yarn to 8 stitches with sugar 'n cream.  It should make nice dishcloths but they will be slightly different from the ones made with sugar 'n cream.  If I would try to use the exact same pattern it will turn out smaller, but if you are a make it up as I go along stitcher like me when I am making dishcloths you will be fine as you will make it the necessary number of stitches to make the size you want.<br />And this is definitely more like a sample pack.  If 

In [62]:
new_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, new_llm)

new_ensemble_answer = new_rag_chain.invoke(query9)
print(new_ensemble_answer)

Based on the provided reviews, I would recommend looking into the product with ASIN: B0BWSF1HPY, which is a soft extra cotton washable tube bulky giant yarn. One of the reviews mentions that this yarn feels like what you'd use to make sweaters or something similar, indicating that it may be suitable for making clothes. However, another review mentions that the product with ASIN: B086HL42Y5 is decent quality but not soft, and wouldn't recommend it for wearable items.

Considering this information, I would suggest exploring the product with ASIN: B0BWSF1HPY further to see if it meets your needs for making clothes.
